# 🌊 Developing Inventory Information for Seattle, WA

## Based on notebook originally for Berkeley, by...

**Authors:** Justin Bonus¹, Pedro Arduino², Barbaros Cetiner²³  

**Institution:** NHERI SimCenter, University of California 

**Affiliations:** ¹ *Shannon and Wilson, Seattle* ; ² *University of Washington, Seattle* ; ³ *University of California, Berkeley*

---

This notebook demonstrates how to assemble a preliminary structural-inventory dataset for the Berkeley region, CA, using the BRAILS-PlusPlus toolkit and SimCenter resources. It (1) defines the study region’s boundary, (2) harvests corresponding building footprints via the OvertureMapsFootprintScraper, (3) enriches those footprints with available structural attributes from the NHERI SimCenter NSI database, and (4) performs basic cleaning—computing centroids, selecting essential fields, and projecting geometries—before visualizing the results on an interactive map. The curated inventory is then exported to a csv file for subsequent hazard evaluation using the Celeris Boussinesq solution toolkit  ...

---

### 🔗 Useful Resources

| Tool/Data Source | Description | Link |
|------------------|-------------|------|
| 🌐 **[BRAILS](https://nheri-simcenter.github.io/BRAILS-Documentation/)** | Tool to generate building inventories from OpenStreetMap and ML methods | [📄 Docs](https://nheri-simcenter.github.io/BRAILS-Documentation/) |

# Check Python Version (must be > 3.10)

Seems to also require < 3.13

In [ ]:
!python --version # shows version from bash shell running jupyter lab, not necessarily for kernel!

In [ ]:
#!pip install --upgrade pip
#!pip install numpy -q
#!pip install plotly -q
#!pip install folium -q
#!pip install git+https://github.com/NHERI-SimCenter/BrailsPlusPlus -q

In [ ]:
#!pip install contextily -q
#!pip install geopandas -q

## Import Packages

In [ ]:
import os
from copy import deepcopy

import numpy as np
import pandas as pd
import requests

import geopandas as gpd
from shapely.geometry import shape
import contextily as ctx
import folium

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from PIL import Image

from brails import Importer

## Step 1: Create a Region Boundary for the Region of Interest
The Importer utility dynamically loads supported data classes. In this step, we define a region boundary using a location name. This boundary polygon will serve as the spatial filter for retrieving building footprints.

In [ ]:
# Define query location:
name_key = 'Seattle'

# File where the building inventory will be stored:
outputfile = '{}_BuildingInventory.geojson'.format(name_key)

# Length units for the attributes (used when relevant):
lengthunit = 'm' # Options are 'm' or 'ft'

In [ ]:
# importer = Importer()

# region_data = {"type": "locationName", "data": "Seattle, WA"}
# region_boundary_class = importer.get_class("RegionBoundary")
# region_boundary_object = region_boundary_class(region_data)

In [ ]:
importer = Importer()

LOCATION_TYPE = 'locationPolygon'
LOCATION_BBOX = (-122.36, 47.62, -122.32, 47.58)

region_data = {'type': LOCATION_TYPE, 'data': LOCATION_BBOX}
region_boundary_object = importer.get_class('RegionBoundary')(region_data)

## Step 2: Import OvertureMapsFootprintScraper and List Available Releases
Next, we load the OvertureMapsFootprintScraper class using the Importer utility. This class allows retrieval of building footprints from Overture Maps. We can also list available Overture Maps releases to choose a specific dataset version.

In [ ]:
overture_scraper_class = importer.get_class("OvertureMapsFootprintScraper")
#_ = overture_scraper_class.fetch_release_names(print_releases=True) # Optional

## Step 3: Retrieve Building Footprints from Overture Maps
We now create an instance of OvertureMapsFootprintScraper using a specific Overture Maps release. Using the previously defined region boundary, we retrieve all building footprints within that area.

Note: If no release is specified, the scraper will use the latest Overture Maps release by default.

In [ ]:
overture_scraper = overture_scraper_class(input_dict={"overtureRelease": "2026-01-21.0"})
overture_inventory = overture_scraper.get_footprints(region_boundary_object)

## Step 4: Save the Retrieved Building Footprints to a GeoJSON File
After retrieving the building footprints, we can export the inventory to a GeoJSON file for further analysis or visualization.

In [ ]:
nsi_scraper = importer.get_class('NSI_Parser')()

In [ ]:
nsi_inventory = nsi_scraper.get_raw_data(region_boundary_object)

In [ ]:
merged_inventory = deepcopy(overture_inventory)
merged_inventory.join(inventory_to_join=nsi_inventory,
                       method='GetPointsNearPolygons')
_ = merged_inventory.write_to_geojson(outputfile)

In [ ]:
# Open the newly created inventory
foot = gpd.read_file(outputfile) 

#quick inspect footprint and inventory
print('Inventory shape:', foot.shape)
print('Inventory columns:', foot.columns)

# Plot the inventory
%matplotlib inline

f, ax = plt.subplots(figsize=(8,8))
foot.plot(ax=ax, color = 'red')
ax.set_title('Building Footprint {0}'.format(name_key))
ax.set_xlabel('Easting')
ax.set_ylabel('Northing')

#Add basemap OpenStreetMap
ctx.add_basemap(ax=ax, crs = foot.crs, source=ctx.providers.OpenStreetMap.Mapnik)
f.savefig(f'{name_key}_Inventory.png')

In [ ]:
# Inspect dataframe
foot.head()

## Step 5: Get Centroids

In [ ]:
# Cleaning dataframe to use 'geometry' attribute
foot.to_crs('EPSG:4326')
foot_c = foot[['id', 'geometry']].copy()

# get extent area centroid
area_c = foot_c.to_crs('EPSG:32610').union_all().centroid
area_c_xy = (area_c.y,area_c.x)

# Project footprint geometry to a planar crs so that centroid calulation is correct
foot_c = foot_c.to_crs('EPSG:32610')

# extract footprint centroids
foot_c['geometry'] = foot_c.geometry.centroid

# Re-project back to original crs
foot_c = foot_c.to_crs('EPSG:4326')

# Getting coordinates lat,lon
foot['Longitude'] = foot_c['geometry'].x
foot['Latitude'] = foot_c['geometry'].y
foot_c['lat_lon'] = list(zip(foot.Latitude, foot.Longitude))

# Coordinates list to street images
coordinates_list = list(foot_c['lat_lon'])
print(coordinates_list[:3])

# id list to street images
id_list = list(foot_c['id'])
print(id_list[:3])

In [ ]:
# Test Plot for Centroids
%matplotlib inline

f, ax = plt.subplots(figsize=(8,16))
foot.plot(ax=ax, color = 'red')
foot_c.plot(ax=ax, color = 'blue', markersize=1)
ax.set_title('Building Footprint {0}'.format(name_key))
ax.set_xlabel('Easting')
ax.set_ylabel('Northing')

#Add basemap OpenStreetMap
ctx.add_basemap(ax=ax, crs = foot.crs, source=ctx.providers.OpenStreetMap.Mapnik)

## Step 6: Save Inventory

In [ ]:
import functools

# clean footprints (if necessary - remove NA columns)
# foot_clean = foot[['id','type', 'fpAreas', 'Longitude', 'Latitude', 'geometry']].copy()
foot_clean = foot[['id','type', 'sqft', 'x', 'y', 'geometry']].copy()

inv_final = foot_clean
inv_final.head(10)

In [ ]:
out_csv = f'{name_key}_Inventory.csv'
inv_final.to_csv(out_csv, index=False)